# Notebook 13: PINN — Universal ODE (Physics-Informed Neural Network) Mode

**Goal:** Understand how `phoscrosstalk.pinn` replaces mechanistic parameter fitting with a physics-informed neural network.

## 1. What is PINN Mode?

In standard mechanistic fitting we optimise θ so the ODE solution matches data. In **PINN (Universal ODE) mode**:

- A neural network `NN(y, t) → correction` is added to the ODE right-hand side
- The NN is trained to satisfy both the data loss **and** the ODE physics loss
- `pinn.enabled = true` bypasses mechanistic multi-start fitting entirely

**Key difference from NeuralODE (Notebook 12):** PINN *replaces* mechanistic fitting; NeuralODE *refines* it afterward.

## 2. Architecture: PINNAugmentation

```
PINNAugmentation
  input     : (state y ∈ ℝ^state_dim, time t)
  output    : correction vector ∈ ℝ^state_dim
  state_dim : 3*K + M + N   (R_rna, S, A, K_dyn, p)
  hidden    : [width_size] * depth  (tanh default)
  gate      : learned scale — inactive dims → zero
  output    : bounded by ±output_clamp (default 10.0)
```

The mechanistic ODE RHS is augmented:

$$\dot{y} = f_{\mathrm{mech}}(y, t;\,\theta) + \mathrm{NN}(y, t)$$

## 3. PINN Loss

$$\mathcal{L} = \mathcal{L}_{\mathrm{data}} + \lambda_{\mathrm{pinn}} \cdot \mathcal{L}_{\mathrm{physics}}$$

$$\mathcal{L}_{\mathrm{data}} = w_p\|P_{\mathrm{sim}}-P_{\mathrm{data}}\|^2 + w_a\|A_{\mathrm{sim}}-A_{\mathrm{data}}\|^2$$

$$\mathcal{L}_{\mathrm{physics}} = \sum_{t_c} \left\|\dot{y}_{\mathrm{NN}}(t_c) - f_{\mathrm{mech}}(y_{\mathrm{NN}}(t_c), t_c)\right\|^2$$

Collocation points `t_c` are spread across the time window (denser than observed points). `regularize = "residual_l2"` uses L2 penalty on ODE residuals.

## 4. `[pinn]` Config Section

```toml
[pinn]
enabled        = false       # true to activate
width_size     = 64
depth          = 2
activation     = "tanh"
lambda_pinn    = 0.1         # physics loss weight
regularize     = "residual_l2"
max_steps      = 500
learning_rate  = 1e-3
optimizer      = "adam"
```

## 5. Key Differences from Mechanistic Fitting

| Aspect | Mechanistic | PINN |
|--------|------------|------|
| Parameter vector θ | Fitted | Not directly used |
| Multi-start | Yes | No (single run) |
| Optimiser | LM / BFGS | Adam |
| Physics enforcement | Exact (ODE integrated) | Soft penalty |
| Interpretability | High | Low |
| Use when | Default | Mechanistic fails |

## 6. PINN Bundle Format

After training, saved in `outdir/pinn_bundle/`:

| File | Contents |
|---|---|
| `pinn_model.eqx` | Equinox model weights |
| `pinn_bundle_meta.json` | `state_dim, K, M, N, width_size, depth, activation, output_clamp` |
| `theta_opt.npy` | Mechanistic θ used during training |

```python
from phoscrosstalk.pinn.outputs import load_pinn_model_bundle
bundle = load_pinn_model_bundle(outdir / 'pinn_bundle')
pinn_model = bundle['pinn_model']   # PINNAugmentation
theta_opt  = bundle['theta_opt']    # np.ndarray
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print("Setup complete.")

In [ ]:
from phoscrosstalk.data_loader import load_site_data, load_kinase_site_matrix

TIMEPOINTS = list(range(1, 15))
sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / 'protephospho.csv'), TIMEPOINTS)
K_site_kin, kinases = load_kinase_site_matrix(str(SAMPLE_DIR / 'kinase_sites.tsv'), sites)

K = len(proteins);  M = len(kinases);  N = len(sites)
state_dim = 3 * K + M + N

print(f'K={K}, M={M}, N={N}')
print(f'Full ODE state_dim = 3*K + M + N = {state_dim}')

## 7. Inspect PINNAugmentation

In [ ]:
import jax
import jax.numpy as jnp

from phoscrosstalk.pinn.model import PINNAugmentation

key = jax.random.PRNGKey(0)
pinn = PINNAugmentation(
    state_dim=state_dim,
    width_size=16,   # small for demo
    depth=2,
    activation='tanh',
    output_clamp=10.0,
    key=key,
)
print('PINNAugmentation created')
print(f'  state_dim    : {pinn.state_dim}')
print(f'  output_clamp : {pinn.output_clamp}')

In [ ]:
# Forward pass: (state_vector, time) → correction of shape (state_dim,)
y_test  = jnp.zeros(state_dim)
t_test  = jnp.array(1.0)
correction = pinn(y_test, t_test)
print(f'Correction shape : {correction.shape}')  # (state_dim,)
print(f'Correction range : [{float(correction.min()):.4f}, {float(correction.max()):.4f}]')
print(f'(Untrained — bounded by output_clamp={pinn.output_clamp})')

## 8. Visualise Untrained PINN Corrections Over Time

In [ ]:
t_grid = jnp.linspace(float(t_phos[0]), float(t_phos[-1]), 80)

corrections = jax.vmap(lambda t: pinn(jnp.zeros(state_dim), t))(t_grid)  # (80, state_dim)
corr_np = np.array(corrections)
t_np    = np.array(t_grid)

labels  = ['R_rna', 'S (mRNA→prot)', 'A (protein)', 'K_dyn (kinase)', 'p (phosphosite)']
slices  = [slice(0,K), slice(K,2*K), slice(2*K,3*K), slice(3*K,3*K+M), slice(3*K+M,state_dim)]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, lbl, sl in zip(axes, labels, slices):
    block = corr_np[:, sl]
    for i in range(block.shape[1]):
        ax.plot(t_np, block[:, i], alpha=0.8)
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel('Time', fontsize=8)
    ax.set_ylabel('NN correction', fontsize=8)
plt.suptitle('Untrained PINNAugmentation corrections (zero state)', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '13_pinn_corrections_untrained.png', dpi=120)
plt.show()
print('Saved: 13_pinn_corrections_untrained.png')

## 9. When to Use PINN Mode

**Use PINN when:**
- Mechanistic fitting consistently fails (very high residuals)
- The ODE structure is wrong but data still has meaningful signal
- You want a data-driven baseline to compare against mechanistic fit

**Limitations:**
- Less interpretable: no per-kinase or per-site parameters
- `lambda_pinn` tuning is critical: too low → ignores physics, too high → underfits
- Single training run: local minima more likely than multi-start mechanistic fitting